---
## 1. Conexión con Azure Machine Learning
Configuramos el cliente de Azure ML y las importaciones necesarias.
**⚠️ Importante:** Esta celda debe ejecutarse SIEMPRE primero. Sin ella, el resto de celdas fallan porque necesitan `mlclient`.

In [ ]:
import os
import mlflow
from azure.ai.ml import MLClient
from azure.ai.ml import automl
from azure.ai.ml import Input
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

print("⏳ Cargando variables de entorno desde el archivo .env...")
load_dotenv()

if not os.getenv("AZURE_WORKSPACE_NAME"):
    raise ValueError("❌ Error: No se encontraron las variables en el archivo .env. Revisa el archivo o la ruta.")

print(f"✅ Variables cargadas. Conectando al Workspace: '{os.getenv('AZURE_WORKSPACE_NAME')}'...")

# Crear el cliente de Azure ML (se usará en todas las celdas siguientes)
mlclient = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)

# ⚠️ NO usar mlflow.set_tracking_uri() cuando se usa azureml-mlflow
# El plugin azureml-mlflow ya configura la conexión automáticamente.
# Si se asigna manualmente, da conflicto con azureml_artifacts_builder.

print(f"\n✅ Cliente Azure ML creado correctamente: {mlclient.workspace_name}")
print("💡 El tracking de MLflow se configura automáticamente vía azureml-mlflow")

⏳ Cargando variables de entorno desde el archivo .env...
✅ Variables cargadas. Conectando al Workspace: 'ml-project-demo'...


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔗 Configurando el URI de seguimiento de MLflow desde variable de entorno...

✅ Cliente Azure ML creado correctamente: ml-project-demo


---
## 2. Estructura de datos MLTable

```text
data/
└── test-data/
    ├── diabetes-test.csv
    └── MLTable
```

In [36]:
# Registrar el dataset como recurso MLTable en Azure
# Si ya existe, lo reutiliza; si no, lo crea
try:
    dataset = mlclient.data.get(
        name="diabetes-dataset", 
        version="1"
        )
    print(f"♻️ Dataset 'diabetes-dataset:1' ya existe. Reutilizando...")
except:
    dataset = Data(
        name="diabetes-dataset",
        version="1",
        description="Dataset de diabetes para AutoML",
        path="./data/test-data",
        type=AssetTypes.MLTABLE
    )
    mlclient.data.create_or_update(dataset)
    print("✅ Dataset 'diabetes-dataset:1' registrado en Azure ML")

♻️ Dataset 'diabetes-dataset:1' ya existe. Reutilizando...


---
## 3. Configurar y ejecutar AutoML
Creamos un **Input** que apunta al dataset ya registrado en la nube y lanzamos el experimento de clasificación.

La columna objetivo es `Diabetic` (0 = no diabético, 1 = diabético).

In [37]:
# Crear la referencia de entrada al dataset registrado en Azure
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:diabetes-dataset:1"
)

# Configurar la tarea de clasificación
job = automl.classification(
    experiment_name="diabetes-automl",
    training_data=my_training_data_input,  # ← Usamos el Input, NO el objeto Data
    target_column_name="Diabetic",
    compute="cpu-cluster",
    primary_metric="accuracy"
)

job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

# Lanzar el experimento
returned_job = mlclient.jobs.create_or_update(job)
print(f"🚀 Job AutoML iniciado: {returned_job.name}")
print(f"📌 Sigue el progreso en Azure ML Studio → Jobs → {returned_job.name}")

🚀 Job AutoML iniciado: sad_turtle_lwfhn42cn6
📌 Sigue el progreso en Azure ML Studio → Jobs → sad_turtle_lwfhn42cn6


---
## 4. Verificación
Comprobamos que el dataset está registrado y el estado del job AutoML.

In [38]:
# Verificar dataset registrado
registered_data = mlclient.data.get(name="diabetes-dataset", version="1")
print(f"📦 Dataset: {registered_data.name} v{registered_data.version}")

# Mostrar estado del job
print(f"📊 Estado del job: {returned_job.status}")
print("🔗 Ve a https://ml.azure.com → Experiments → diabetes-automl para ver el progreso")

📦 Dataset: diabetes-dataset v1
📊 Estado del job: NotStarted
🔗 Ve a https://ml.azure.com → Experiments → diabetes-automl para ver el progreso


---
## 5. (Opcional) Ejecutar job por comandos con job.yml
También puedes lanzar el entrenamiento directamente desde el `job.yml` ya configurado.

In [39]:
from azure.ai.ml import load_job
from azure.ai.ml.entities import Job

# Cargar job desde el archivo YAML
job_from_yml = load_job(source="./src/job.yml")

# Lanzar el job
returned_job_yml = mlclient.jobs.create_or_update(job_from_yml)
print(f"🚀 Job desde YAML iniciado: {returned_job_yml.name}")
print(f"📌 Ver en Azure ML Studio → Jobs → {returned_job_yml.name}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.01 MBs): 100%|███████

ResourceNotFoundError: (UserError) No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 
Code: UserError
Message: No environment exists for name: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu, version: 

---
# Entrenamiento y seguimiento de modelos en cuadernos
## Creación de un experimento de MLflow

Como viene en la [guía de MLflow](https://mlflow.org/docs/latest/ml/tracking/#automatic-logging?azure-portal=true) podemos usar esta herramienta:
---
- ### mlflow.start_run()  
sirve para crear un experimento de MLflow y registrar métricas y parámetros de entrenamiento. Por ejemplo:
```python
with mlflow.start_run():
    mlflow.log_param("lr", 0.001)
    # Your ML code
    ...
    mlflow.log_metric("val_loss", val_loss)
```
---

- ### mlflow.autolog() - Habilitación del registro automático

para que te monte todo el seguimiento de métricas y parámetros automáticamente (tiene soporte con scikit-learn, TensorFlow, PyTorch, XGBoost, LightGBM y Keras).mlflow.autolomlflow.autolog()

---
- ### mlflow.xgboost.autolog() - Utilice el registro personalizado
Para registrar de forma manual y NO de forma automática, puedes usar `mlflow.xgboost.autolog()` para registrar métricas y parámetros de entrenamiento de modelos XGBoost.

MLflow inicia una ejecución dentro de un experimento en Azure Machine Learning para empezar a realizar el seguimiento de la ejecución.

In [1]:
mlflow.set_experiment(experiment_name="diabetes-automl")
# Sino le pondria el nombre "default"

NameError: name 'mlflow' is not defined

In [ ]:
# Cargar datos del CSV y dividir en train/test
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("./data/test-data/diabetes-test.csv")
print(f"📊 Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

X = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
        'SerumInsulin','BMI','DiabetesPedigree','Age']].values
y = df['Diabetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
print(f"✅ Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

📊 Datos cargados: 5000 filas, 9 columnas
✅ Train: 3500 samples | Test: 1500 samples


In [40]:
# Ejemplo de Registro Automatico de MLflow con XGBoost
from xgboost import XGBClassifier

with mlflow.start_run():
    mlflow.autolog()

    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

2026/07/09 18:40:31 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/07/09 18:40:31 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/07/09 18:40:36 ERROR mlflow.xgboost: Failed to log feature importance plot. XGBoost autologging will ignore the failure and continue. Exception: 
Traceback (most recent call last):
  File "/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/mlflow/xgboost/__init__.py", line 738, in train_impl
    log_feature_importance_plot(features, importance, imp_type)
  File "/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/mlflow/xgboost/__init__.py", line 637, in log_feature_importance_plot
    mlflow.log_artifact(filepath)
  File "/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/mlflow/tracking/fluent.py", line 1659, in log_artifact
    import mlflow
  File "/home/renato/Desktop/MLOps-practicas/azure_mlflow

🏃 View run jovial_beard_qtlbxz24 at: https://spaincentral.api.azureml.ms/mlflow/v2.0/subscriptions/51808865-b3b4-45b4-b26d-a8bbefd234b5/resourceGroups/VisualStudioOnline-E5B9097B7CB94C978336EF055E22167E/providers/Microsoft.MachineLearningServices/workspaces/ml-project-demo/#/experiments/89dd0b15-9760-47e5-a6df-0c140d1f66cc/runs/a5a70d51-32e8-466c-95f7-68358880a46d
🧪 View experiment at: https://spaincentral.api.azureml.ms/mlflow/v2.0/subscriptions/51808865-b3b4-45b4-b26d-a8bbefd234b5/resourceGroups/VisualStudioOnline-E5B9097B7CB94C978336EF055E22167E/providers/Microsoft.MachineLearningServices/workspaces/ml-project-demo/#/experiments/89dd0b15-9760-47e5-a6df-0c140d1f66cc


In [3]:
#Ejemplo de Registro Manual de MLflow con XGBoost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

with mlflow.start_run():
    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

NameError: name 'mlflow' is not defined